In [1]:
import zipfile
import os

zip_path = "./archive.zip"   # change filename
extract_path = "./"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done")


Done


In [1]:
!pip install opencv-python tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 5.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0+nv25.2 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.4.2 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.4.2 which is incompatible.
scipy 1.12.0 requires numpy<1.29.0,>=1.22.4, but you have numpy 2.4.2 which is incompatible.
numba 0.59.1 requires numpy<1.27,>=1.22, but you have numpy 2.4.2 which is incompatible.


In [3]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

Found existing installation: numpy 2.4.2
Uninstalling numpy-2.4.2:
  Successfully uninstalled numpy-2.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 5.7 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [4]:
!pip uninstall opencv-python -y
!pip install opencv-python==4.8.1.78

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 5.6 MB/s eta 0:00:0000:0100:01


In [5]:
!apt-get update
!apt-get install -y libgl1


Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease 
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1857 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2240 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [38.1 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [3196 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2016 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1207 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [3381 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [34.8 kB]
Get:13 http://arc

In [6]:
import cv2
print("OpenCV version:", cv2.__version__)


OpenCV version: 4.8.1


In [4]:
import os

BASE_PATH = "/LAB"

IMAGE_DIR = f"{BASE_PATH}/train-image/images"
LABEL_DIR = f"{BASE_PATH}/train-image/labels"
DATA_YAML = f"{BASE_PATH}/train-image/data.yaml"
RUNS_DIR = f"{BASE_PATH}/runs"

os.makedirs(RUNS_DIR, exist_ok=True)


In [8]:
import cv2
import os
from tqdm import tqdm

BASE_PATH = "/LAB"

IMAGE_DIR = f"{BASE_PATH}/train-image/images"
LABEL_DIR = f"{BASE_PATH}/train-image/labels"
BOXED_DIR = f"{BASE_PATH}/boxed_images"


os.makedirs(LABEL_DIR, exist_ok=True)
os.makedirs(BOXED_DIR, exist_ok=True)

for img_name in tqdm(os.listdir(IMAGE_DIR)):
    if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join(IMAGE_DIR, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    h, w, _ = img.shape

    # Preprocessing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, thresh = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        continue

    c = max(contours, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(c)

    # YOLO format
    x_center = (x + bw / 2) / w
    y_center = (y + bh / 2) / h
    bw_norm = bw / w
    bh_norm = bh / h

    label_path = os.path.join(
        LABEL_DIR, img_name.rsplit('.', 1)[0] + ".txt"
    )

    with open(label_path, "w") as f:
        f.write(f"0 {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}")

    # Save boxed image
    boxed = img.copy()
    cv2.rectangle(boxed, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
    cv2.imwrite(os.path.join(BOXED_DIR, img_name), boxed)

print("✅ Bounding boxes created successfully")


100%|██████████| 33130/33130 [01:03<00:00, 518.52it/s]

✅ Bounding boxes created successfully


In [5]:
!pip install ultralytics

INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 5.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 5.4 MB/s eta 0:00:0000:0100:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 5.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 5.2 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 5.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 5.3 MB/s eta 0:

In [6]:
from ultralytics import YOLO
print("YOLOv8 Ready")


WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8 Ready


In [11]:
import os

folders = [
    f"{BASE_PATH}/train-image/images/train",
    f"{BASE_PATH}/train-image/images/val",
    f"{BASE_PATH}/train-image/images/test",
    f"{BASE_PATH}/train-image/labels/train",
    f"{BASE_PATH}/train-image/labels/val",
    f"{BASE_PATH}/train-image/labels/test"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Folder structure created under /LAB")



✅ Folder structure created under /LAB


In [12]:
import os
import random
import shutil


IMG_DIR = f"{BASE_PATH}/train-image/images"
LBL_DIR = f"{BASE_PATH}/train-image/labels"

image_files = []
for f in os.listdir(IMG_DIR):
    if f.lower().endswith((".jpg", ".jpeg", ".png")):
        image_files.append(f)

print("Total images found:", len(image_files))

valid_pairs = []
for img in image_files:
    name = os.path.splitext(img)[0]
    label_path = os.path.join(LBL_DIR, name + ".txt")
    if os.path.exists(label_path):
        valid_pairs.append(img)

print("Valid image-label pairs:", len(valid_pairs))

# 🔴 STOP if zero
if len(valid_pairs) == 0:
    raise ValueError("No valid image-label pairs found. Check label filenames.")

random.shuffle(valid_pairs)

n = len(valid_pairs)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

splits = {
    "train": valid_pairs[:train_end],
    "val": valid_pairs[train_end:val_end],
    "test": valid_pairs[val_end:]
}

for split, files in splits.items():
    for img in files:
        base = os.path.splitext(img)[0]

        shutil.copy(
            os.path.join(IMG_DIR, img),
            f"{BASE_PATH}/train-image/images/{split}/{img}"
        )
        shutil.copy(
            os.path.join(LBL_DIR, base + ".txt"),
            f"{BASE_PATH}/train-image/labels/{split}/{base}.txt"
        )

print("✅ Dataset split completed successfully")


Total images found: 33126
Valid image-label pairs: 33126
✅ Dataset split completed successfully


In [14]:
%%writefile /LAB/train-image/data.yaml
path: /LAB//train-image

train: images/train
val: images/val
test: images/test

names:
  0: skin_lesion



Overwriting /LAB/train-image/data.yaml


In [21]:
from ultralytics import YOLO

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov8s.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov8s"
)


New https://pypi.org/project/ultralytics/8.4.12 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.11 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 325.3±157.6 MB/s, size: 17.3 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.2Git/s 0.0s
Plotting labels to /LAB/runs/yolov8s/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 256 train, 256 val
Using 2 dataloader workers
Logging results to /LAB/runs/yolov8s
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50       2.5G     0.8749      1.161      1.213         17        256: 100% ━━━━━━━━━━━━ 941/941 2.5it/s 6:15<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 1.1s/it 

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      2.91G     0.4789     0.5746     0.9802          6        256: 100% ━━━━━━━━━━━━ 941/941 2.6it/s 6:02<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 1.0it/s 2:170.4ss
                   all       9150       9150      0.878       0.86      0.948      0.865

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      2.91G      0.465     0.5609     0.9743          6        256: 100% ━━━━━━━━━━━━ 941/941 3.3it/s 4:44<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 1.7s/it 4:081.8ss
                   all       9150       9150       0.88      0.861      0.949      0.867

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      2.91G     0.4581     0.5499     0.9677          6     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x764b3755e930>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [22]:
from ultralytics import YOLO

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov8l.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov8l"
)

New https://pypi.org/project/ultralytics/8.4.12 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.11 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 480.6±245.8 MB/s, size: 17.3 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 913.8Mit/s 0.0s
Plotting labels to /LAB/runs/yolov8l/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Image sizes 256 train, 256 val
Using 2 dataloader workers
Logging results to /LAB/runs/yolov8l
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      6.47G     0.8488      1.169      1.286         17        256: 100% ━━━━━━━━━━━━ 941/941 1.3it/s 11:38<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 1.2

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      7.82G     0.4407     0.5407      1.029          6        256: 100% ━━━━━━━━━━━━ 941/941 3.7it/s 4:14<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 4.3it/s 33.3s0.2s
                   all       9150       9150      0.896       0.87      0.956      0.889

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      7.82G     0.4296     0.5265      1.021          6        256: 100% ━━━━━━━━━━━━ 941/941 3.7it/s 4:17<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 4.3it/s 33.3s0.2ss
                   all       9150       9150        0.9      0.868      0.957       0.89

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      7.82G     0.4226      0.515      1.013          6    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x764b44a71e80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [23]:
from ultralytics import YOLO

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov8m.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov8m"
)

New https://pypi.org/project/ultralytics/8.4.12 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.11 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 451.5±204.2 MB/s, size: 17.3 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.1Git/s 0.0s
Plotting labels to /LAB/runs/yolov8m/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 256 train, 256 val
Using 2 dataloader workers
Logging results to /LAB/runs/yolov8m
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      5.02G      0.851      1.153      1.255         17        256: 100% ━━━━━━━━━━━━ 941/941 3.7it/s 4:18<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 4.9it/s 

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      5.62G     0.4586     0.5587      1.032          6        256: 100% ━━━━━━━━━━━━ 941/941 5.1it/s 3:04<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.0it/s 28.6s0.2s
                   all       9150       9150      0.898      0.863      0.954      0.879

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      5.62G     0.4458      0.543      1.022          6        256: 100% ━━━━━━━━━━━━ 941/941 5.1it/s 3:04<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.0it/s 28.6s0.2s
                   all       9150       9150      0.896      0.866      0.955      0.881

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      5.62G     0.4343     0.5323      1.015          6     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x764b27f417f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
from ultralytics import YOLO

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov9s.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov9s"
)

Ultralytics 8.4.12 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov9s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov9s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      3.54G     0.4765      0.576      1.064          6        256: 100% ━━━━━━━━━━━━ 941/941 2.8it/s 5:33<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.7it/s 25.1s0.2s
                   all       9150       9150      0.901      0.851      0.951      0.872

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      3.54G      0.462     0.5596      1.054          6        256: 100% ━━━━━━━━━━━━ 941/941 2.9it/s 5:30<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.8it/s 24.7s0.2s
                   all       9150       9150      0.903      0.852      0.952      0.873

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      3.54G      0.456     0.5483      1.045          6     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x794289192630>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [9]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()
# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov9m.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=16,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=True,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov9m"
)

Ultralytics 8.4.12 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov9m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov9m, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      1.84G     0.4707     0.5804      1.052          6        256: 100% ━━━━━━━━━━━━ 1881/1881 5.1it/s 6:12<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 286/286 9.4it/s 30.4s<0.1s
                   all       9150       9150       0.89      0.844      0.944      0.864

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      1.84G     0.4568     0.5653      1.035          6        256: 100% ━━━━━━━━━━━━ 1881/1881 5.0it/s 6:15<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 286/286 8.3it/s 34.4s<0.1s
                   all       9150       9150      0.895      0.843      0.945      0.865

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      1.84G     0.4482     0.5571      1.028          

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b975757b830>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [17]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov10s.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov10s"
)

New https://pypi.org/project/ultralytics/8.4.13 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.12 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 191.1±40.4 MB/s, size: 17.3 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 556.2Mit/s 0.0s
Plotting labels to /LAB/runs/yolov10s/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 99 weight(decay=0.0), 112 weight(decay=0.0005), 111 bias(decay=0.0)
Image sizes 256 train, 256 val
Using 2 dataloader workers
Logging results to /LAB/runs/yolov10s
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      3.04G     0.9104      1.489      1.226         17        256: 100% ━━━━━━━━━━━━ 941/941 3.0it/s 5:18<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.2

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      3.46G     0.5215     0.6242     0.9973          6        256: 100% ━━━━━━━━━━━━ 941/941 4.5it/s 3:27<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 6.0it/s 23.8s0.2s
                   all       9150       9150      0.844      0.785      0.897      0.797

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      3.46G     0.5081     0.6048     0.9879          6        256: 100% ━━━━━━━━━━━━ 941/941 4.2it/s 3:42<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 5.3it/s 27.0s0.2s
                   all       9150       9150      0.847      0.786        0.9        0.8

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/50      3.46G     0.4957     0.5986     0.9805          6     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b977d6e6f00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [13]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov9c.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov9c"
)

Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov9c.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov9c3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, p

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 500.8±400.9 MB/s, size: 17.3 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.2Git/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 154 weight(decay=0.0), 161 weight(decay=0.0005), 160 bias(decay=0.0)
Plotting labels to /LAB/runs/yolov9c3/labels.jpg... 
Image sizes 256 train, 256 val
Using 2 dataloader workers
Logging results to /LAB/runs/yolov9c3
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
WARNING ⚠️ CUDA out of memory with batch=32. Reducing to batch=16 and retrying (1/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 20.4±43.5 MB/s, size: 15.3 KB)
train: Scanning /LAB/train-image/labels/train.cache... 30086 images, 0 ba

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 698.8±555.1 MB/s, size: 20.5 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.2Git/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 154 weight(decay=0.0), 161 weight(decay=0.0005), 160 bias(decay=0.0)
: 0% ──────────── 0/941  1.9s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
WARNING ⚠️ CUDA out of memory with batch=16. Reducing to batch=8 and retrying (2/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 296.5±302.0 MB/s, size: 21.8 KB)
train: Scanning /LAB/train-image/labels/train.cache... 30086 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30086/30086 11.5Git/s 0.0s


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 284.2±179.4 MB/s, size: 18.5 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.5Git/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 154 weight(decay=0.0), 161 weight(decay=0.0005), 160 bias(decay=0.0)
: 0% ──────────── 0/1881  2.2s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
WARNING ⚠️ CUDA out of memory with batch=8. Reducing to batch=4 and retrying (3/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 42.3±88.2 MB/s, size: 19.0 KB)
train: Scanning /LAB/train-image/labels/train.cache... 30086 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30086/30086 11.5Git/s 0.0s


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 27.2±30.6 MB/s, size: 18.6 KB)
val: Scanning /LAB/train-image/labels/val.cache... 9150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9150/9150 1.3Git/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 154 weight(decay=0.0), 161 weight(decay=0.0005), 160 bias(decay=0.0)
: 0% ──────────── 0/3761  2.1s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
: 0% ──────────── 0/7522  0.1s


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 8.50 MiB is free. Process 5188 has 17.11 GiB memory in use. Process 176638 has 1.48 GiB memory in use. Process 176953 has 1.48 GiB memory in use. Process 183268 has 894.00 MiB memory in use. Process 185170 has 1.48 GiB memory in use. Process 194697 has 1.10 GiB memory in use. Of the allocated memory 887.02 MiB is allocated by PyTorch, and 2.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [11]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov10m.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov10m"
)

Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:2 (NVIDIA RTX A5000, 24122MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/LAB/train-image/data.yaml, degrees=0.0, deterministic=True, device=2, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/LAB/yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov10m3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 2.50 MiB is free. Process 5188 has 17.11 GiB memory in use. Process 176638 has 1.48 GiB memory in use. Process 176953 has 1.48 GiB memory in use. Process 183268 has 894.00 MiB memory in use. Process 185170 has 1.48 GiB memory in use. Process 194697 has 1.11 GiB memory in use. Of the allocated memory 894.72 MiB is allocated by PyTorch, and 1.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolov10l.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolov10l"
)

In [ ]:
from ultralytics import YOLO
import torch 


torch.cuda.empty_cache()

# Load model from LAB (or current dir if already there)
model = YOLO("/LAB/yolo11s.pt")

model.train(
    data="/LAB/train-image/data.yaml",  # ✅ absolute LAB path
    epochs=50,
    imgsz=256,
    batch=32,          # 🔽 reduced for OOM safety
    device=0,         # GPU
    workers=2,        # 🔽 reduce CPU-GPU overhead
    amp=False,        # 🔽 disable mixed precision
    cache=False,      # 🔥 VERY IMPORTANT
    project="/LAB/runs",
    name="yolo11s"
)